#1. Descripción y objetivo

En el presente notebook se realiza el EDA de un CSV extraído de Kaggle con datos sobre la página web Filmaffinity, con calificaciones de diverso contenido audiovisual. El objetivo del análisis de estos datos es extraer conclusiones sobre los títulos con mejor nota y determinar si esta se ve influida por parámetros como el género o el año.

#2. Carga de las librerias y del CSV

In [ ]:
# En primer lugar se cargan las librerías, tanto pandas y numpy como las librerías de visualización

import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

In [ ]:
# A continuación se carga el CSV, ya subido a los archivos de Colab, y obtenemos con él nuestro df

df = pd.read_csv("filmaffinity_dataset.csv")

#3. Exploración inicial

In [ ]:
# Hacemos una pequeña exploración del df ya cargado

df.head()

,Unnamed: 0,Título,Año,País,Dirección,Reparto,Nota,Tipo filme,Género
0,0,'49-'17,1917,Estados Unidos,Ruth Ann Baldwin,"Joseph W. Girard, Leo Pierson, William Dyer, M...",--,Película,Western
1,1,"10,000 Years B.C. (C)",1916,Estados Unidos,Willis H. O'Brien,NaN,"5,1",Cortometraje,Comedia
2,2,1812,1912,Rusia,"Vasili Goncharov, Kai Hansen, Aleksandr Uralsky","Pavel Knorr, Vasili Goncharov, Aleksandra Gonc...",--,Película,Drama
3,3,20.000 leguas de viaje submarino (C),1907,Francia,Georges Méliès,Georges Méliès,"6,0",Cortometraje,Fantástico
4,4,A Bad Case (C),1909,Francia,Émile Cohl,NaN,"5,3",Cortometraje,Comedia


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119003 entries, 0 to 119002
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  119003 non-null  int64 
 1   Título      119003 non-null  object
 2   Año         119003 non-null  int64 
 3   País        119003 non-null  object
 4   Dirección   118202 non-null  object
 5   Reparto     94525 non-null   object
 6   Nota        119003 non-null  object
 7   Tipo filme  119003 non-null  object
 8   Género      119003 non-null  object
dtypes: int64(2), object(7)
memory usage: 8.2+ MB


Observamos que tenemos más de 100.000 filas y que, entre los tipos de filme, hay también cortometrajes, por ejemplo. Será necesario investigar dicha columna con mayor detalle para determinar los distintos tipos de contenido y elegir cuáles utilizar. Igualmente, vemos que únicamente la columna de año, aparte de "Unnamed: 0" aparecen ya en formato numérico, pero no así la de Nota, por lo que será necesario hacer transformaciones en dicha columna.

# 4. Limpieza de datos

## Renombrado de columnas

In [ ]:
# La columna "Unnamed: 0" es poco descriptiva e intuitiva para poder trabajar con ella,
# por lo que se cambia su nombre a "Id" al observar que simplemente contiene un ranking numéricos de los títulos

df.rename(columns = {"Unnamed: 0": "Id"}, inplace = True)

In [ ]:
df.head()

,Id,Título,Año,País,Dirección,Reparto,Nota,Tipo filme,Género
0,0,'49-'17,1917,Estados Unidos,Ruth Ann Baldwin,"Joseph W. Girard, Leo Pierson, William Dyer, M...",--,Película,Western
1,1,"10,000 Years B.C. (C)",1916,Estados Unidos,Willis H. O'Brien,NaN,"5,1",Cortometraje,Comedia
2,2,1812,1912,Rusia,"Vasili Goncharov, Kai Hansen, Aleksandr Uralsky","Pavel Knorr, Vasili Goncharov, Aleksandra Gonc...",--,Película,Drama
3,3,20.000 leguas de viaje submarino (C),1907,Francia,Georges Méliès,Georges Méliès,"6,0",Cortometraje,Fantástico
4,4,A Bad Case (C),1909,Francia,Émile Cohl,NaN,"5,3",Cortometraje,Comedia


## Filtrado de columnas

In [ ]:
# Observamos distintos de contenido en "Tipo filme". Nos fijamos en la columna
# y verificamos con mayor detalle el tipo de contenidos que contiene

df["Tipo filme"].value_counts()

,count
Tipo filme,
Película,73651
Cortometraje,16407
Serie,10498
Estreno televisivo,8516
Documental,8499
Miniserie,1432


In [ ]:
# Para este análisis únicamente queremos analizar el tipo películas, así que vamos
# a quedarnos solamente con dicho contenido

df = df[df["Tipo filme"] == "Película"]

In [ ]:
# Ahora que la columna "Tipo filme" solo contiene películas, ha dejado de aportarnos información relevante,
# por lo que se elimina del df

df.drop("Tipo filme", axis=1, inplace=True)

In [ ]:
# Observamos que muchos títulos no incluyen la nota, y vienen indicados con el texto "--".
# Al ser esto un texto estático en lugar de un valor nulo, no es posible usar el métoto drop.na,
# por lo que se usará una expresión similar a la anterior para conservar solo el tipo de filme película,
# pero esta vez, al contrario, excluyendo todos los valores con el texto "--""

df = df[df["Nota"] != "--"]

In [ ]:
# Verificamos de nuevo los datos

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 41329 entries, 23 to 118999
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         41329 non-null  int64 
 1   Título     41329 non-null  object
 2   Año        41329 non-null  int64 
 3   País       41329 non-null  object
 4   Dirección  41192 non-null  object
 5   Reparto    38767 non-null  object
 6   Nota       41329 non-null  object
 7   Género     41329 non-null  object
dtypes: int64(2), object(6)
memory usage: 2.8+ MB


,Id,Título,Año,País,Dirección,Reparto,Nota,Género
23,23,A Fool There Was,1915,Estados Unidos,Frank Powell,"Theda Bara, Edward José, Mabel Frenyear, Runa ...","5,8","Drama, Romance"
68,68,A prueba de balas,1917,Estados Unidos,John Ford,"Harry Carey, Duke R. Lee, George Berrell, Moll...","6,1",Western
79,79,Al sol,1919,Estados Unidos,Charles Chaplin,"Charles Chaplin, Edna Purviance, Tom Wilson, T...","6,7",Comedia
93,93,Amalia,1914,Argentina,Enrique García Velloso,"Susana Larreta y Quintana, Luis García Lawson,...","4,2",Drama
107,107,Ana Bolena,1920,Alemania,Ernst Lubitsch,"Emil Jannings, Hedwig Pauly-Winterstein, Hilde...","6,6",Drama


In [ ]:
# Vemos cómo hemos pasado de tener más de 100.000 registros a menos de 50.000, reduciéndose
# el tamaño total del df en más de la mitad

In [ ]:
# Convertimos la columna "Nota" a tipo numérico. Para ello, primero convertimos la coma decimal
# (correcta en formato español) a punto para que Python la interprete correctamente como decimal
# y a continuación hacemos la conversión a float

df["Nota"] = df["Nota"].str.replace(",", ".", regex=False).astype(float)

In [ ]:
# Del mismo modo, convertimos ID a tipo texto, por ser una columna que no vamos
# a analizar numéricamente. La columna año podría convertirse también a texto, pero
# de momento se mantendrá, ya que se utilizará para hacer más transformaciones
df["Id"] = df["Id"].astype(str)

In [ ]:
# Hacemos ahora un describe para obtener datos rápidamente sobre las variables numéricas
df.describe()

,Año,Nota
count,41329.000000,41329.000000
mean,1990.483365,5.428551
std,24.320011,1.237490
min,1906.000000,1.300000
25%,1973.000000,4.600000
50%,1998.000000,5.600000
75%,2011.000000,6.300000
max,2020.000000,9.000000


Con simplemente este describe, ya podemos disponer de información valiosa sobre los datos:

- Las películas abarcan desde 1906 hasta 2020, situándose la media en 1990 y la mediana en 1998. Al ser la mediana superior a la media, ya podemos saber que tendremos más películas recientes que antiguas (lo cual es coherente), por llegarnos más producciones actuales que recientes

- Las notas abarcan desde un mínimo de 1,3 hasta un máximo de 9, con una mediana de 5,43 y una media de 5,6. Al tener valores similares ambos valores, podemos determinar que, en principio, no hay demasiados valores atípicos o outliers. De hecho, incluso el Q1 (4,6) y el Q3 (6,3) se encuentran cercanos a la mediana, lo que indicaría que los valores están concentrados mayormente en esta escala media.

In [ ]:
# Vamos a revisar ahora si hay columnas con valores nulos

df.isnull().sum()

,0
Id,0
Título,0
Año,0
País,0
Dirección,137
Reparto,2562
Nota,0
Género,0


In [ ]:
# Observamos que "Dirección" y "Reparto" tienen un cierto número de nulos. Echemos un vistazo
# a estos nulos para saber si vale la pena mantenerlos o no:

df[df.isnull().any(axis=1)].head(50)

,Id,Título,Año,País,Dirección,Reparto,Nota,Género
573,573,El último malón,1917,Argentina,Alcides Greca,NaN,5.8,Drama
686,686,Hacia la luz,1919,Dinamarca,NaN,"Asta Nielsen, Augusta Blad, Frederik Jacobsen,...",5.7,Drama
1986,1986,"Vida, nacimiento y muerte de Cristo",1906,Francia,Alice Guy-Blaché,NaN,5.8,Drama
2692,2692,Blancanieves y los siete enanitos,1937,Estados Unidos,David Hand,NaN,6.8,Animación
2952,2952,Chang: A Drama of the Wilderness,1927,Estados Unidos,"Merian C. Cooper, Ernest B. Schoedsack",NaN,6.6,Aventuras
2955,2955,Chanson d'Armor,1934,Francia,Jean Epstein,NaN,7.5,"Drama, Romance"
3755,3755,El cuento del zorro,1937,Francia,"Wladyslaw Starewicz, Irene Starewicz",NaN,7.2,"Animación, Comedia"
4525,4525,Festival de los 11 Premios de Walt Disney,1937,Estados Unidos,"Burt Gillett, Wilfred Jackson, David Hand",NaN,6.8,"Animación, Comedia"
4677,4677,Garras de oro,1926,Colombia,P.P. Jambrina,NaN,6.4,Drama
4918,4918,Hierba,1925,Estados Unidos,"Merian C. Cooper, Ernest B. Schoedsack",NaN,7.3,Aventuras


Tras ver los datos, comprobamos que la gran mayoría de "nulos" en reparto se debe a películas de Animación, lo cual tiene sentido, por lo que se mantiene. En cuanto a dirección, se observan valores nulos, pero, al no poder imputar valores fácilmente, se mantienen teniéndolos en cuenta como limitación

In [ ]:
#Veamos ahora datos estadísticos agrupados por país:

df.groupby("País").describe()

Año                                                   \
                 count         mean        std     min      25%     50%   
País                                                                      
 del Este (RDA)   14.0  1967.857143  12.377754  1946.0  1960.75  1967.0   
Afganistán         2.0  2007.500000   6.363961  2003.0  2005.25  2007.5   
Albania            1.0  2008.000000        NaN  2008.0  2008.00  2008.0   
Alemania         624.0  1994.003205  31.743606  1911.0  1997.00  2007.0   
Andorra            1.0  1999.000000        NaN  1999.0  1999.00  1999.0   
...                ...          ...        ...     ...      ...     ...   
a y Herzegovina    7.0  2007.571429   6.704654  1997.0  2003.50  2010.0   
blica del Congo    1.0  2010.000000        NaN  2010.0  2010.00  2010.0   
del Oeste (RFA)  233.0  1972.935622   9.566022  1951.0  1965.00  1973.0   
ia y Montenegro    4.0  2003.500000   0.577350  2003.0  2003.00  2003.5   
oviética (URSS)  241.0  1963.800830  19.600429  1924.0  1956.00  1967.0   

                                   Nota                                        \
                     75%     max  count      mean       std  min    25%   50%   
País                                                                            
 del Este (RDA)  1974.50  1989.0   14.0  6.464286  0.767220  4.8  6.225  6.65   
Afganistán       2009.75  2012.0    2.0  7.000000  0.282843  6.8  6.900  7.00   
Albania          2008.00  2008.0    1.0  6.500000       NaN  6.5  6.500  6.50   
Alemania         2014.00  2020.0  624.0  5.520353  1.236621  1.3  4.600  5.70   
Andorra          1999.00  1999.0    1.0  3.600000       NaN  3.6  3.600  3.60   
...                  ...     ...    ...       ...       ...  ...    ...   ...   
a y Herzegovina  2011.50  2016.0    7.0  6.600000  0.616441  5.5  6.350  6.70   
blica del Congo  2010.00  2010.0    1.0  5.700000       NaN  5.7  5.700  5.70   
del Oeste (RFA)  1981.00  1990.0  233.0  5.980687  1.074515  2.5  5.200  6.20   
ia y Montenegro  2004.00  2004.0    4.0  6.800000  0.416333  6.3  6.600  6.80   
oviética (URSS)  1979.00  1991.0  241.0  6.727801  0.720022  3.4  6.300  6.80   

                            
                  75%  max  
País                        
 del Este (RDA)  6.90  7.3  
Afganistán       7.10  7.2  
Albania          6.50  6.5  
Alemania         6.50  8.3  
Andorra          3.60  3.6  
...               ...  ...  
a y Herzegovina  6.95  7.4  
blica del Congo  5.70  5.7  
del Oeste (RFA)  6.80  8.0  
ia y Montenegro  7.00  7.3  
oviética (URSS)  7.20  8.3  

[138 rows x 16 columns]

Este describe también arroja información relevante, por ejemplo:

- Observamos que muchos nombres de países aparecen truncados por la izquierda
- Hay países con un número muy reducido de películas (menos de 10). Por motivos de representatividad, estos países también se eliminarán

In [ ]:
# Obtenemos primero un recuento completo por país:

recuento = df["País"].value_counts()

In [ ]:
# Y ahora conservamos solo aquellos países con al menos 10 resultados:

paises_frecuentes = recuento[recuento > 10].index
df = df[df["País"].isin(paises_frecuentes)]

In [ ]:
# Observamos que ya no tenemos resultados con menos de 10 películas

df.groupby("País").describe()

Año                                                   \
                  count         mean        std     min      25%     50%   
País                                                                       
 del Este (RDA)    14.0  1967.857143  12.377754  1946.0  1960.75  1967.0   
Alemania          624.0  1994.003205  31.743606  1911.0  1997.00  2007.0   
Argentina        1031.0  1992.332687  22.451434  1914.0  1976.00  1999.0   
Australia         371.0  2003.264151  13.813244  1906.0  1994.00  2008.0   
Austria            69.0  1997.507246  25.458515  1922.0  1992.00  2008.0   
...                 ...          ...        ...     ...      ...     ...   
Venezuela          40.0  2001.625000  14.497900  1950.0  1994.00  2006.5   
Vietnam            15.0  2004.733333  11.398413  1984.0  1997.00  2006.0   
Yugoslavia         49.0  1981.428571  10.845506  1958.0  1975.00  1982.0   
del Oeste (RFA)   233.0  1972.935622   9.566022  1951.0  1965.00  1973.0   
oviética (URSS)   241.0  1963.800830  19.600429  1924.0  1956.00  1967.0   

                                    Nota                                  \
                     75%     max   count      mean       std  min    25%   
País                                                                       
 del Este (RDA)  1974.50  1989.0    14.0  6.464286  0.767220  4.8  6.225   
Alemania         2014.00  2020.0   624.0  5.520353  1.236621  1.3  4.600   
Argentina        2012.00  2020.0  1031.0  5.327158  1.285528  1.8  4.300   
Australia        2014.00  2020.0   371.0  5.213208  1.113365  1.8  4.500   
Austria          2014.00  2020.0    69.0  5.891304  1.078226  1.7  5.400   
...                  ...     ...     ...       ...       ...  ...    ...   
Venezuela        2012.25  2019.0    40.0  5.890000  0.762519  3.5  5.600   
Vietnam          2015.00  2019.0    15.0  5.946667  1.093400  3.7  5.300   
Yugoslavia       1989.00  2001.0    49.0  6.622449  0.609735  4.3  6.400   
del Oeste (RFA)  1981.00  1990.0   233.0  5.980687  1.074515  2.5  5.200   
oviética (URSS)  1979.00  1991.0   241.0  6.727801  0.720022  3.4  6.300   

                                   
                  50%    75%  max  
País                               
 del Este (RDA)  6.65  6.900  7.3  
Alemania         5.70  6.500  8.3  
Argentina        5.50  6.300  8.1  
Australia        5.30  6.100  8.1  
Austria          6.10  6.600  7.6  
...               ...    ...  ...  
Venezuela        6.05  6.325  7.2  
Vietnam          6.10  6.900  7.2  
Yugoslavia       6.70  6.900  7.8  
del Oeste (RFA)  6.20  6.800  8.0  
oviética (URSS)  6.80  7.200  8.3  

[64 rows x 16 columns]

## Limpieza de columnas

In [ ]:
# A continuación, vamos a modificar los nombres de países para que se muestren correctamente.
# Para ello, primero obtenemos el resultado completo de países:

df["País"].unique()

array(['Estados Unidos', 'Argentina', 'Alemania', 'Italia', 'Dinamarca',
       'Rusia', 'México', 'Francia', 'Suecia', 'Serbia', 'Rumanía',
       'Japón', 'Bélgica', 'India', 'España', 'Australia', 'Reino Unido',
       'Portugal', 'oviética (URSS)', 'Colombia', 'Checoslovaquia',
       'Chile', 'Brasil', 'Bajos (Holanda)', 'China', 'Noruega',
       'Finlandia', 'Suiza', 'Austria', 'Hungría', 'Polonia', 'Bolivia',
       'Grecia', 'del Oeste (RFA)', ' del Este (RDA)', 'Egipto',
       'Yugoslavia', 'Cuba', 'Venezuela', 'Israel', 'Corea del Sur',
       'Filipinas', 'Irlanda', 'Irán', 'Taiwán', 'Perú', 'Canadá',
       'Senegal', 'Hong Kong', 'Turquía', 'Bulgaria', 'Sudáfrica',
       'Nueva Zelanda', 'Islandia', 'Tailandia', 'República Checa',
       'Uruguay', 'Vietnam', 'Estonia', 'Indonesia', 'Rep. Dominicana',
       'Líbano', 'Malasia', 'Singapur'], dtype=object)

In [ ]:
#Hacemos un diccionario con los países con nombres incorrectos

diccionario_paises = {
    "oviética (URSS)": "URSS",
    "del Oeste (RFA)": "RFA",
    " del Este (RDA)": "RDA",
    "Bajos (Holanda)": "Holanda"
}

In [ ]:
# Y con replace cambiamos los resultados en cuestión

df["País"] = df["País"].replace(diccionario_paises)

In [ ]:
# Por último, previamente también se mostró que la columna "Género" contenía,
# en ciertas ocasiones, más de un valor. Esto supone un problema al tomarse cada una
# de estas opciones como valores distintos, lo que llevaría a análisis incompletos
# o erróneos.

df["Género"] = df["Género"].str.split(',').str[0]

In [ ]:
# Verificamos los resultados finales con unique

df["Género"].unique()

array(['Drama', 'Western', 'Comedia', 'Romance', 'Intriga', 'Thriller',
       'Aventuras', 'Acción', 'Animación', 'Terror', 'Fantástico',
       'Ciencia ficción', 'Infantil'], dtype=object)

## Columna de década

Para poder facilitar el manejo del análisis temporal y agrupar datos que permitan llegar a conclusiones más relevantes, se agruparán los años de producción por década. De este modo, en lugar de tener los datos de años aislados, se facilitará el análisis entre décadas, como los 70 y los 90, en lugar de año por año, que no tiene tanta representatividad.

In [ ]:
# Para ello, utilizamos una división

df["Década"] = df["Año"]// 10 * 10

In [ ]:
df.head()

,Id,Título,Año,País,Dirección,Reparto,Nota,Género,Década
23,23,A Fool There Was,1915,Estados Unidos,Frank Powell,"Theda Bara, Edward José, Mabel Frenyear, Runa ...",5.8,Drama,1910
68,68,A prueba de balas,1917,Estados Unidos,John Ford,"Harry Carey, Duke R. Lee, George Berrell, Moll...",6.1,Western,1910
79,79,Al sol,1919,Estados Unidos,Charles Chaplin,"Charles Chaplin, Edna Purviance, Tom Wilson, T...",6.7,Comedia,1910
93,93,Amalia,1914,Argentina,Enrique García Velloso,"Susana Larreta y Quintana, Luis García Lawson,...",4.2,Drama,1910
107,107,Ana Bolena,1920,Alemania,Ernst Lubitsch,"Emil Jannings, Hedwig Pauly-Winterstein, Hilde...",6.6,Drama,1920


# Visualizaciones

In [ ]:
# En primer lugar vamos a observar con un histograma cómo se distribuyen las notas de las películas:

fig = px.histogram(
    df,
    x="Nota",
    nbins=20,
    color_discrete_sequence=["yellow"]
)
fig.update_traces(marker_line_color="blue", marker_line_width=3)
fig.update_layout(title="Distribución de notas")
fig.show()

Observamos una distribución estándar ligeramente hacia la izquierda. Los grandes bloques de distribución se encuentran entre el 5,3 y el 6,7. Lo que llama mucho la atención es cómo, precisamente a partir de ese 6,7 hay una disminución brusca al siguiente bloque, de 6,8 a 7,2 y aún más brusca al posterior, de 7,3 a 7,7, con bastante menos de la mitad de resultados que el bloque anterior.
La conclusión de este histograma es que la audiencia vota de un modo relativamente uniforme, otorgando mayormente puntuaciones de 5 a 7, dejando el 8, el 9 y 10 como notas generalmente excepcionales. Ocurre algo similar en el lado inverso de la tabla, con el 4 siendo aún relativamente frecuente, pero con notas inferiores al 4 siendo bastante menos frecuentes.

In [ ]:
# A continuación vamos a plasmar lo que puede influir el país de producción en la nota:

nota_por_pais = (
    df.groupby("País")["Nota"]
    .agg(Nota_media="mean", Recuento="count")
    .round(2)
    .sort_values("Recuento", ascending=False)
    .head(20)
    .reset_index()
)
nota_por_pais

,País,Nota_media,Recuento
0,Estados Unidos,5.25,17829
1,España,4.88,3959
2,Reino Unido,5.64,2942
3,Francia,6.03,2739
4,Japón,6.07,2280
5,Italia,5.47,2021
6,Argentina,5.33,1031
7,Canadá,4.71,862
8,México,5.36,791
9,Alemania,5.52,624


In [ ]:
# De la tabla ya tenemos datos, interesantes, pero primero vamos a hacer
# un gráfico para poder visualizar mejor la media:

fig = px.bar(
    nota_por_pais,
    x="Nota_media",
    y="País",
    orientation="h",
    color="Nota_media",
    color_continuous_scale="YlGnBu",
    title="Nota media por país de producción"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False
)

fig.show()

Conclusiones: Observamos que el primer país es la URSS, con un 6,73, lo que supone una diferencia importante sobre el segundo puesto (Suecia, con 6,14). De hecho, los 3 países por debajo de Suecia aún obtienen una media superior al 6. Teniendo en cuenta el total de producciones, que se observa en la tabla, Estados Unidos es el ganador claro, representando él solo casi la mitad de las producciones (casi 18.000 frente a 41.000 en total). Sin embargo, su nota media se sitúa en la parte inferior del ranking, con un 5,25 de nota media. Una hipótesis precisamente es que, al tener mayor producciones, algunas de ellas no cuenten con tanto favor de público, mientras que de la URSS, con un número bastante inferior, solo se voten los títulos más relevantes. El caso de España es más acusado aún, con casi 4.000 producciones, pero una media suspensa (4,88). Estos resultados se acompañarán de análisis posteriores relacionados con el año/década de producción y el género para poder extraer más conclusiones.

In [ ]:
nota_por_decada = (
    df.groupby("Década")["Nota"]
    .agg(Nota_media="mean", Recuento="count")
    .round(2)
    .sort_values("Recuento", ascending=False)
    .head(20)
    .reset_index()
)
nota_por_decada

,Década,Nota_media,Recuento
0,2010,5.10,11112
1,2000,5.20,7952
2,1990,5.37,4978
3,1980,5.32,4130
4,1970,5.54,3484
5,1960,5.81,3255
6,1950,6.10,2648
7,1940,6.30,1466
8,1930,6.37,1053
9,2020,5.17,463


Observamos, al ordenar por recuento, cómo hay una progresión bastante gradual en el número de títulos según la década. De hecho, la única excepción es 2020, pero hay que tener en cuenta que el csv solo contiene datos del propio año 2020 y ninguno más de dicha década, lo cual es una limitación de la que se parte que se debe tener en cuenta.

In [ ]:
fig = px.bar(
    nota_por_decada,
    x="Nota_media",
    y="Década",
    orientation="h",
    color="Nota_media",
    color_continuous_scale="YlGnBu",
    title="Nota media por década de producción"
)


fig.update_layout(
    yaxis={"categoryorder": "total descending"},
    coloraxis_showscale=False
)

fig.show()

Conclusiones: Se observa cómo, de manera general, la nota media disminuye progresivamente con el tiempo, salvo por ciertas excepciones. Así, la década con mayor nota es la de 1920, con un 6,69 de media, a la que seguiría la de 1930, ya con 3 décimas menos (6,37). En cambio, observamos cómo la década con menor nota es la de 2010, con un aprobado justito (5,1). De nuevo, el tener un mayor número de producciones de diversa calidad y no únicamente obras que hayan perdurado en el tiempo puede ser motivo de esta diferencia, pero también es necesario analizar la media por géneros.

In [ ]:
# Como curiosidad, vamos a crear una matriz de correlación entre las variables numéricas
# poder determinar si de verdad hay una correlación negativa fuerte.
df.corr(numeric_only=True)

,Año,Nota,Década
Año,1.000000,-0.301761,0.993149
Nota,-0.301761,1.000000,-0.301348
Década,0.993149,-0.301348,1.000000


In [ ]:
# Observamos que, aunque hay correlación negativa, no es tan fuerte (-0,3).
# Vamos a probar ahora con otro método, como el de Spearman.
df[["Año", "Nota", "Década"]].corr(method="spearman")


,Año,Nota,Década
Año,1.000000,-0.276219,0.984436
Nota,-0.276219,1.000000,-0.277914
Década,0.984436,-0.277914,1.000000


La correlación de Spearman también arroja una correlación negativa, pero igualmente relativa (-0,28 en este caso). Esto indica que, aunque a priori se puede ver cierto sesgo general por la distribución de las medias, hay ciertas diferencias que pueden impedir una distribución negativa total, como que la media de 2020 sea algo más alta que la de 2010, o la de 1990 sobre la de 1980.

In [ ]:
# Finalmente, vamos a hacer una agrupación por género

nota_por_genero = (
    df.groupby("Género")["Nota"]
    .agg(Nota_media="mean", Recuento="count")
    .round(2)
    .sort_values("Recuento", ascending=False)
    .reset_index()
)
nota_por_genero

,Género,Nota_media,Recuento
0,Drama,6.15,13642
1,Comedia,5.18,5905
2,Acción,4.93,5375
3,Terror,4.25,3447
4,Aventuras,5.36,2096
5,Animación,5.77,2008
6,Intriga,5.45,1718
7,Thriller,5.13,1579
8,Ciencia ficción,4.83,1566
9,Romance,5.32,1501


In [ ]:
fig = px.bar(
    nota_por_genero,
    x="Nota_media",
    y="Género",
    orientation="h",
    color="Nota_media",
    color_continuous_scale="YlGnBu",
    title="Nota media por género"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False
)

fig.show()

Conclusiones: A pesar de ser el género más numeroso, "Drama" es el que tiene mayor media. Por tener más votaciones, los outliers tienen menor peso, lo que hace que sería probable que ocupara un lugar central en la tabla, pero no es el caso, siendo el único género con una calificación media superior a 6. Esto puede atribuirse al "prestigio" que se asocia al género.

Sorprende un tanto, en cambio, el segundo puesto de "Animación", a veces relegado como un género menor. Será interesante analizarlo en mayor detalle cruzándolo con otras variables. Recordemos que Japón era un país con una elevada nota media, y se trata de un país con una larga tradición en el género de Animación.

Contrario a lo que podría parecer, el género "Comedia" se encuentra justo en el centro de la tabla, con una media de 5,18. Aunque suele ser un género algo denostado, la valoración media no es tan baja, así que, de nuevo, será interesante cruzarlo con otras variables (en este caso, quizá la de década).

En cuanto a las últimas posiciones, no sorprende tanto que las ocupen "Terror" e "Infantil", aunque quizá sí la diferencia sobre el resto (hay más de medio punto de diferencia entre "Terror" y el siguiente género, "Ciencia ficción").

In [ ]:
# A continuación vamos a hacer un boxplot para determinar mejor la distribución por género

fig = px.box(
    df,
    x="Género",
    y="Nota",
    title="Distribución de media por género"
)

fig.show()

Observamos que, efectivamente, "Drama", "Terror" e "Infantil" son los tres géneros con una distribución más distinta al resto. "Drama" tiene realmente un alto número de outliers en la parte inferior, debido también a contar con una elevada cifra de producciones, pero se compensa con un suelo superior (Q1) al del resto de géneros. En cambio, "Terror" presente ya outliers a partir del  7,2, siendo esta una cifra aún relativamente normal en el resto de géneros, salvo por "Infantil", que tiene una distribución casi idéntica a "Terror", pero sin apenas outliers. Se observa también cómo "Terror" ocupa el mínimo total con un 1,3, seguido de cerca por "Acción" con un 1,4.

In [ ]:
# A continuación vamos a hacer un scatter plot que tenga en cuenta el recuento
# de producciones y la nota media por país en el género "Animación" para los 10 países
# con mayor número de producciones

nota_por_animacion_pais = (
    df[df["Género"] == "Animación"].groupby("País")["Nota"]
    .agg(Nota_media="mean", Recuento="count")
    .round(2)
    .sort_values("Recuento", ascending=False).
    head(10)
    .reset_index()
)
nota_por_animacion_pais

,País,Nota_media,Recuento
0,Estados Unidos,5.55,695
1,Japón,6.08,634
2,Francia,6.29,119
3,España,5.25,96
4,Reino Unido,6.15,91
5,Canadá,5.12,38
6,Alemania,4.84,36
7,Argentina,5.01,27
8,México,5.11,23
9,Italia,5.09,21


In [ ]:
fig = px.scatter(
    nota_por_animacion_pais,
    x="Nota_media",
    y="País",
    size="Recuento",
    color="Nota_media",
    color_continuous_scale="Viridis",
    title="Distribución de notas de animación por país"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False
)

fig.show()

Observamos que, en cierto modo, Japón sí que está contribuyendo a aumentar la media de "Animación", por ser uno de los países con mayor nota media y, además, uno de los mayores productores del género, encontrándose de hecho a poca distancia de Estados Unidos. Sin embargo, no es desdeñable tampoco la aportación de Reino Unido y, sobre todo, de Francia, con medias más elevadas aún. Cabe destacar que Francia, aunque a menor escala, como se puede observar, es un país que también posee una larga tradición en el género de animación, lo que explicaría estos datos.

In [ ]:
# Veamos por último el cruce de comedias por década:

nota_por_decada_comedia = (
    df[df["Género"] == "Comedia"].groupby("Década")["Nota"]
    .agg(Nota_media="mean", Recuento="count")
    .round(2)
    .sort_values("Recuento", ascending=False)
    .reset_index()
)
nota_por_decada_comedia

,Década,Nota_media,Recuento
0,2010,5.04,1083
1,2000,4.93,1070
2,1990,5.00,842
3,1980,4.90,727
4,1960,5.34,574
5,1970,4.76,513
6,1950,5.84,420
7,1940,6.04,258
8,1930,6.30,221
9,1920,6.70,75


In [ ]:
fig = px.scatter(
    nota_por_decada_comedia,
    x="Nota_media",
    y="Década",
    size="Recuento",
    color="Nota_media",
    color_continuous_scale="Viridis",
    title="Distribución de notas de comedia por década"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False
)

fig.show()

Observamos cómo, efectivamente, hay una diferencia muy alta entre el periodo anterior y posterior a la década de los 60. Curiosamente, la década con la nota más baja es la de los 70, aunque las notas de este periodo se encuentran en una horquilla similar, del 4,76 al 5,04. La década de los 60 se encuentra aislada en un punto medio mientras que las anteriores a los 60 abarcan del 5,84 de 1950 al 6,7 de 1920.

Sin embargo, a pesar de que vemos diferencias acusadas por décadas, la media total de comedia era de 5,18, por lo que la diferencia entre décadas tampoco influye significativamente en la nota global. Esto se debe al altísimo número de producciones en particular en la década de los 2000 y de 2010, que hacen que tengan un mayor peso sobre el total.

# Conclusiones generales

- De manera general, la distribución de las notas se encuentra en un punto medio, del 5,3 al 6,7. Sin embargo, vimos también que se forma una especie de "techo de cristal" a partir del 6,7 y, en especial, a partir del 8, con medias ya muy inalcanzables en el conjunto.
- Hay ciertas diferencias notables por género y por década, siendo "Drama" y la primera mitad del siglo XX los tipos más favorecidos, mientras que "Terror" e "Infantil" y el periodo posterior a los 70 incluido siendo las opciones con nota más baja.
- En cuanto a los países, podría observarse cierta correlación negativa entre el recuento de producciones y la nota, es decir, que a mayor número de títulos, peor media en general, lo cual puede atribuirse a sesgo de títulos "exportables" de países extranjeros (no se incluye aquí a Estados Unidos por su hegemonía).

Finalizado ya el EDA de estos datos en Python, puede procederse a la exportación del csv manipulado para profundizar con el análisis en detalle en Power BI.

In [ ]:
df.to_csv("filmaffinity_clean.csv", index=False)